# ImmunoTherapy — Baseline Model: Logistic Regression

Predicting **irAE occurrence** (`Grade 2+` = 1 vs `Grade 0-1` = 0) from the Khan
*JITC* 2025 cohort: 146 patients, 40 baseline cytokines + demographics + ANA titer.

This is a deliberately **basic baseline** — L2-regularized logistic regression with
standard preprocessing, evaluated with repeated stratified CV. No log-transform or
feature selection yet (those are the next iteration). The goal is an honest read on
how much signal is here before adding complexity.

## 1. Load data
The repo is public, so we read the CSV straight from GitHub — no upload needed.

In [ ]:
import pandas as pd
import numpy as np

RAW_URL = "https://raw.githubusercontent.com/vishnusrikant/ImmunoTherapy/main/ActionableData/Khan_workable.csv"
df = pd.read_csv(RAW_URL)

print("shape:", df.shape)
df.head(3)

## 2. Sanity check
Confirm the target is clean 0/1 and the cytokines loaded as numeric.

In [ ]:
TARGET = "iraE_occurrence"
print("target balance:")
print(df[TARGET].value_counts(), "\n")
print("dtypes summary:")
print(df.dtypes.value_counts())
print("\nmissing values per column (only showing >0):")
print(df.isna().sum()[df.isna().sum() > 0])

## 3. Define features & build the pipeline

- **Numeric** (40 cytokines + 2 ANA columns): median-impute (ANA is ~62% missing) → standardize.
- **Categorical** (gender, ethnicity, race, cancer_type, ici_drug): one-hot encode.
- **Model**: `LogisticRegression` (L2, the sklearn default) with `max_iter` raised so it converges.

Everything lives inside one `Pipeline` so preprocessing is **refit inside each CV fold** —
no leakage of scaler/imputer statistics from validation into training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

ID_COL   = "patient_id"
CAT_COLS = ["gender", "ethnicity", "race", "cancer_type", "ici_drug"]
ANA_COLS = ["ana_titer_baseline", "ana_titer_missing"]
# everything else (besides id/target/cat/ana) is a cytokine measurement
CYTO_COLS = [c for c in df.columns if c not in [ID_COL, TARGET] + CAT_COLS + ANA_COLS]
NUM_COLS  = CYTO_COLS + ANA_COLS

print(f"{len(CYTO_COLS)} cytokines + {len(ANA_COLS)} ANA = {len(NUM_COLS)} numeric, "
      f"{len(CAT_COLS)} categorical")

X = df[NUM_COLS + CAT_COLS]
y = df[TARGET].astype(int)

numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
categorical = OneHotEncoder(handle_unknown="ignore")

pre = ColumnTransformer([
    ("num", numeric,     NUM_COLS),
    ("cat", categorical, CAT_COLS),
])

clf = Pipeline([
    ("pre", pre),
    ("lr",  LogisticRegression(max_iter=5000, random_state=42)),
])
clf

## 4. Cross-validated performance

With only 146 rows a single train/test split is too noisy to trust, so we use
**RepeatedStratifiedKFold (5 folds × 10 repeats = 50 fits)** and report the mean ± std
of ROC-AUC, PR-AUC (average precision), and accuracy.

Reference points: a no-skill classifier scores **ROC-AUC 0.50**, and always-predicting
the majority class gives **accuracy ≈ 0.54** (79/146).

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

for metric in ["roc_auc", "average_precision", "accuracy"]:
    scores = cross_val_score(clf, X, y, cv=cv, scoring=metric)
    print(f"{metric:18s}: {scores.mean():.3f} +/- {scores.std():.3f}")

## 5. Confusion matrix & classification report
Using out-of-fold predictions (each patient predicted by a model that never saw them).

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(clf, X, y, cv=skf)

print(classification_report(y, y_pred, target_names=["Grade 0-1", "Grade 2+"]))

ConfusionMatrixDisplay.from_predictions(
    y, y_pred, display_labels=["Grade 0-1", "Grade 2+"], cmap="Blues")
plt.title("Out-of-fold confusion matrix")
plt.show()

## 6. ROC curve (out-of-fold probabilities)

In [ ]:
from sklearn.metrics import RocCurveDisplay

y_proba = cross_val_predict(clf, X, y, cv=skf, method="predict_proba")[:, 1]
RocCurveDisplay.from_predictions(y, y_proba, name="Logistic Regression")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="No skill")
plt.title("ROC — out-of-fold")
plt.legend()
plt.show()

## 7. Which features carry the signal?
Refit on all 146 patients and read the standardized coefficients (log-odds). Positive =
pushes toward `Grade 2+`. With correlated cytokines, treat individual coefficients as
*suggestive*, not definitive — that's what the elastic-net iteration will firm up.

In [ ]:
clf.fit(X, y)
feat_names = clf.named_steps["pre"].get_feature_names_out()
coefs = clf.named_steps["lr"].coef_[0]

coef_df = (pd.DataFrame({"feature": feat_names, "coef": coefs})
           .assign(abs_coef=lambda d: d["coef"].abs())
           .sort_values("abs_coef", ascending=False))

print("Top 15 features by |coefficient|:")
coef_df.head(15)[["feature", "coef"]]

## Next steps (deliberately not done here)
1. **Log-transform the cytokines** (`np.log1p`) — they span ~6 orders of magnitude; this
   is the single change most likely to help logistic regression.
2. **Elastic-net penalty** (`penalty="elasticnet", solver="saga", l1_ratio=...`) with a
   CV grid — adds feature selection and stabilizes correlated cytokines.
3. **Compare against HistGradientBoosting** to see the non-linear ceiling.
4. Collapse rare `ici_drug` / `cancer_type` levels to cut one-hot noise.